In [1]:
!pip -q install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.2 MB/s eta 0:00:00


In [9]:
import anthropic

API_KEY = "UwU"

SYSTEM_PROMPT = r"""
You are LLM-A, the first stage of a text-to-federated-SPARQL pipeline.

INPUT: a natural-language question.
OUTPUT: exactly two blocks, in this order, with no text before or after:
1. A SPARQL skeleton
2. A JSON block describing what must be retrieved from the knowledge graphs

FORMAT

```sparql
[skeleton]
```

```json
{
  "properties": { "?var": "description" },
  "classes":    { "?var": "description" },
  "entities":   { "?var": "description" }
}
```

SKELETON RULES
- Write the query as if all data lives in ONE knowledge graph. No SERVICE blocks, no PREFIX declarations, no real IRIs.
- Use `a` for type statements.
- Every KG-specific item (a property, a class, a named entity) must be a placeholder variable. Describe each one in the JSON.
- Plain values that are not KG entities (numbers, years, dates, free-text strings) stay inline in the skeleton, written plainly (5, 2010, "text"). No datatype annotations like ^^xsd:integer. These are NOT listed in the JSON.
- Use any SPARQL construct you need: OPTIONAL, FILTER, GROUP BY, ORDER BY, LIMIT, BIND, VALUES, aggregates.
- Variable names: short, snake_case, descriptive. An instance variable and its class variable must be different (e.g. ?paper and ?paper_type).
- A variable has exactly ONE role. A variable in the predicate position (a property) must NEVER also appear as a subject or object. Name properties like verbs (?bowled_by, ?authored) and nodes like nouns (?bowler, ?paper).
- Triple direction must match your JSON description (subject -> object).

JSON RULES
- Every placeholder variable for a property, class or entity appears in exactly one JSON key, and the name must exactly match the skeleton.
- Every variable in the JSON must appear in the skeleton. Do not declare a property or class the skeleton does not use. If you list a class, the skeleton must contain a triple like `?x a ?x_type .`
- Result and intermediate variables (e.g. ?paper, ?date, ?count) are omitted from the JSON.
- Entities are ONLY specific named things that appear in the question (a person, place, organization, drug, etc.). Variables for things the question asks for are NEVER entities. If the question contains no named entity, omit the "entities" key.
- Property descriptions: natural language, specific, and state the direction, e.g. "links a person (subject) to a paper they authored (object)".
- Class descriptions: natural language, specific.
- Entity descriptions: the name plus disambiguation hints (occupation, nationality, time period, field, known works).
- Descriptions describe ONLY the property, class or entity itself. Never include filter values, constraints or wording from the question.
- If a category has no entries, omit that key.
- If the question cannot be expressed as a query, output `SELECT * WHERE {}` and `{}`.

BEFORE ANSWERING, CHECK
1. No variable is used as both a property and a node.
2. Every variable in the JSON appears in the skeleton.
3. Every property, class and entity placeholder in the skeleton appears in the JSON.
4. No result variable is listed under "entities".

EXAMPLE 1: "Papers by Einstein who worked at Princeton"

```sparql
SELECT ?paper WHERE {
  ?paper a ?paper_type .
  ?einstein ?authored ?paper .
  ?einstein ?worked_at ?princeton .
}
```

```json
{
  "properties": {
    "?authored": "links a person (subject) to a paper they authored (object)",
    "?worked_at": "links a person (subject) to an organization they worked at (object)"
  },
  "classes": {
    "?paper_type": "academic paper or scholarly publication"
  },
  "entities": {
    "?einstein": "Albert Einstein, theoretical physicist (German-American, Nobel laureate)",
    "?princeton": "Princeton University, research university in New Jersey, USA"
  }
}
```

EXAMPLE 2: "How many drugs targeting BRCA1 were approved after 2010?"

```sparql
SELECT (COUNT(DISTINCT ?drug) AS ?count) WHERE {
  ?drug a ?drug_type .
  ?drug ?targets ?brca1 .
  ?drug ?approval_date ?date .
  FILTER(YEAR(?date) > 2010)
}
```

```json
{
  "properties": {
    "?targets": "links a drug (subject) to the gene or protein it targets (object)",
    "?approval_date": "links a drug (subject) to the date it was approved (object)"
  },
  "classes": {
    "?drug_type": "approved pharmaceutical drug"
  },
  "entities": {
    "?brca1": "BRCA1, human gene/protein involved in DNA repair and breast cancer susceptibility"
  }
}
```

EXAMPLE 3 (no named entity): "Which films won more than 3 awards, and who directed them?"

```sparql
SELECT ?film ?director WHERE {
  ?film a ?film_type .
  ?film ?awards_won ?award_count .
  ?film ?directed_by ?director .
  FILTER(?award_count > 3)
}
```

```json
{
  "properties": {
    "?awards_won": "links a film (subject) to the number of awards it won (object)",
    "?directed_by": "links a film (subject) to the person who directed it (object)"
  },
  "classes": {
    "?film_type": "feature film or motion picture"
  }
}
```
"""

def ask(question, model="claude-sonnet-5-5", max_tokens=3000):
    client = anthropic.Anthropic(
        api_key=API_KEY,
        base_url="https://capi.aerolink.lat/",
    )
    msg = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": question}],
    )
    return "".join(b.text for b in msg.content if b.type == "text")

In [10]:
print(ask("Which drugs target proteins that are associated with Alzheimer's disease?"))
print(ask("Papers by Einstein who worked at Princeton"))

```sparql
SELECT ?drug ?protein WHERE {
  ?drug a ?drug_type .
  ?drug ?targets ?protein .
  ?protein ?associated_with ?alzheimers .
}
```

```json
{
  "properties": {
    "?targets": "links a drug (subject) to the protein or gene it targets (object)",
    "?associated_with": "links a protein or gene (subject) to a disease it is associated with (object)"
  },
  "classes": {
    "?drug_type": "pharmaceutical drug or therapeutic compound"
  },
  "entities": {
    "?alzheimers": "Alzheimer's disease, a progressive neurodegenerative disorder"
  }
}
```
```sparql
SELECT ?paper WHERE {
  ?paper a ?paper_type .
  ?einstein ?authored ?paper .
  ?einstein ?worked_at ?princeton .
}
```

```json
{
  "properties": {
    "?authored": "links a person (subject) to a paper they authored (object)",
    "?worked_at": "links a person (subject) to an organization they worked at (object)"
  },
  "classes": {
    "?paper_type": "academic paper or scholarly publication"
  },
  "entities": {
    "?einstein": 

In [11]:
print(ask("Which bowlers have taken 5 wickets in a single Test innings, and which teams did they play for?"))

```sparql
SELECT ?bowler ?team WHERE {
  ?bowler a ?bowler_type .
  ?bowler ?bowling_performance ?performance .
  ?performance ?wickets_in_innings 5 .
  ?bowler ?represents ?team .
}
```

```json
{
  "properties": {
    "?bowling_performance": "links a bowler (subject) to their bowling performance in an innings (object)",
    "?wickets_in_innings": "links a bowling performance (subject) to the number of wickets taken (object)",
    "?represents": "links a cricketer (subject) to the national or domestic team they play for (object)"
  },
  "classes": {
    "?bowler_type": "cricketer who bowls in Test cricket"
  },
  "entities": {}
}
```
